In [1]:
#  your task is to write PySpark code that:

# Reads the student_scores.csv file into a DataFrame.
# Adds a new column called Rank_Within_Subject.
# This new column should rank the students within each subject based on 
# their score in descending order (highest score gets rank 1).
# Shows the resulting DataFrame.

In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

In [3]:
spark = SparkSession.builder.appName("Grades").getOrCreate()

25/04/07 09:35:36 WARN Utils: Your hostname, Nikhils-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.90 instead (on interface en0)
25/04/07 09:35:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/07 09:35:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/07 09:35:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
df = spark.read.csv("grades.csv", header=True, inferSchema=True)

In [5]:
df.printSchema()

root
 |-- Student: string (nullable = true)
 |-- Subject: string (nullable = true)
 |-- Score: integer (nullable = true)



In [6]:
window_spec = Window.partitionBy("Subject").orderBy(desc("Score"))

25/04/07 09:35:54 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
df_added = df.withColumn("Rank_Within_Subject", rank().over(window_spec))

In [8]:
df_added.show()

+-------+-------+-----+-------------------+
|Student|Subject|Score|Rank_Within_Subject|
+-------+-------+-----+-------------------+
|  Alice|English|   90|                  1|
|Charlie|English|   88|                  2|
|    Bob|English|   82|                  3|
|    Bob|   Math|   92|                  1|
|  Alice|   Math|   85|                  2|
|Charlie|   Math|   76|                  3|
|Charlie|Science|   95|                  1|
|    Bob|Science|   88|                  2|
|  Alice|Science|   78|                  3|
+-------+-------+-----+-------------------+



In [9]:
df.createOrReplaceTempView("scores_table")

In [14]:
result_df = spark.sql("select * from scores_table where Subject='Math'")

In [15]:
result_df.show()

+-------+-------+-----+
|Student|Subject|Score|
+-------+-------+-----+
|  Alice|   Math|   85|
|    Bob|   Math|   92|
|Charlie|   Math|   76|
+-------+-------+-----+



In [16]:
agg_df = spark.sql("select Subject, avg(Score) from scores_table group by Subject")

In [17]:
agg_df.show()

+-------+-----------------+
|Subject|       avg(Score)|
+-------+-----------------+
|Science|             87.0|
|   Math|84.33333333333333|
|English|86.66666666666667|
+-------+-----------------+

